Getting VizDoom up and running


In [ ]:
%pip install vizdoom


In [ ]:
%cd github & git clone https://github.com/mwydmuch/ViZDoom

In [ ]:
#import vizdoom for the env
from vizdoom import *
#import random for the action sampling
import random
#import time for sleeping 
import time
#import numpy for identity matrix
import numpy as np

In [ ]:
#Game setup
game = DoomGame()
game.load_config('github/VizDoom/scenarios/basic.cfg')  #to load the specific configuratons
game.init()

In [ ]:
#this is set the actions we can take in the environment
actions = np.identity(3,dtype=np.uint8) #create's a 3*3 identity matrix

In [ ]:

print("left move:",actions[0])

print("right move:",actions[1])
print("attack:",actions[2])


In [ ]:
random.choice(actions)

In [ ]:
# game.new_episode()
# game.is_episode_finished()
#game.make_action(random.choice(actions))

In [ ]:
# state = game.get_state()
# state.screen_buffer  # cotains an numpy array of img 
# info = state.game_variables   #game variable asuch as ammo
# info

In [ ]:
#loop through episodes
episodes=10
for episode in range(episodes):
    #create a new episode or game 
    game.new_episode()
    #check the game isn't done
    while not game.is_episode_finished():
        #get the game state
        state = game.get_state()
        #get the game image
        img = state.screen_buffer
        #get the game variable- Ammo
        info = state.game_variables
        #take an action
        reward = game.make_action(random.choice(actions) ,4 ) #helps to skip the frames while visualizing.
        #print reward
        print('reward :',reward)
        time.sleep(0.02) #time for the break in millli seconds
    print('Result:',game.get_total_reward())
    time.sleep(2)



In [ ]:
game.close()

2.Converting it to the Open Ai Gym wnvironment

In [ ]:
%pip install gym
%pip install opencv-python



In [ ]:
import gymnasium as gym
#import environment base class from the OpenAI gym
from gym import Env
#import gym spaces
from gym.spaces import Discrete,Box
#import opencv
import cv2

In [ ]:
Discrete(3).sample() #random choice selection
actions[Discrete(3).sample()]

In [ ]:
Box(low = 0,high=255,shape=(224,224),dtype=np.uint8).sample()

In [ ]:
#create a base class VizDoom Environment
class VizDoomGym(Env):
    #Funtion that is called when we call the env
    def __init__(self, render=False):
        #inherit form Env
        super().__init__()
        
        #Game setup
        self.game = DoomGame()
        self.game.load_config('github/VizDoom/scenarios/basic.cfg')  #to load the specific configuratons
        
        
        #Render frame logic
        if render==False:
            self.game.set_window_visible(False)
        else:
            self.game.set_window_visible(True)
            
        self.game.init()
        
        #create an action and observation space
        self.observation_space= Box(low=0,high=255,shape=(100,160,1),dtype=np.uint8)
        self.action_space = Discrete(3)
        
        pass
    
    #this is how we take a step in the environment 
    def step(self,action):
        #Specify action and take step
        actions = np.identity(3,dtype=np.uint8) #create's a 3*3 identity matrix
        reward = self.game.make_action(actions[action],4)
        
        if self.game.get_state():
            state=self.game.get_state().screen_buffer
            state = self.grayscale(state)
            ammo = self.game.get_state().game_variables[0]
            info=ammo
        else:
            state = np.zeros(self.observation_space.shape)
            info =0 
        info={"info":info}
        done = self.game.is_episode_finished()
        
        return state,reward,done,info
    
    #Define how to render the game or environment
    def render(self):
        pass
    
    #What happens when we start a new game
    def reset(self):
        self.game.new_episode()
        state =  self.game.get_state().screen_buffer
        return self.grayscale(state)
        
    
    #Grayscale the game frame and resize it
    def grayscale(self, observation):
        gray = cv2.cvtColor(np.moveaxis(observation,0,-1),cv2.COLOR_BGR2GRAY)
        resize = cv2.resize(gray,(160,100),interpolation=cv2.INTER_CUBIC)
        state = np.reshape(resize,(100,160,1))
        return state
    
    #Call to close down the game
    def close(self):
        self.game.close()
        pass

View game state


In [ ]:
env =VizDoomGym(render=True)


In [ ]:
state=env.reset()


In [ ]:
env.step(2)

In [ ]:
env.reset()

In [ ]:
env.close()

In [ ]:
# #import Env checker
# from stable_baselines3.common import env_checker
# env_checker.check_env(env, warn=True)

view state


In [ ]:
%pip install matplotlib

3.Mark down


In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.imshow(cv2.cvtColor(state,cv2.COLOR_BGR2RGB))

Set Callback


In [ ]:
%pip install torch torchvision torchaudio 


In [ ]:
%pip install stable-baselines3

In [ ]:
#Import os for file nav
import os
#import callback class form sb3
from stable_baselines3.common.callbacks import BaseCallback


In [ ]:
class TrainAndLoggingCallback(BaseCallback):

    def __init__(self, check_freq, save_path, verbose=1):
        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(
                self.save_path,
                "best_model_{}".format(self.n_calls)
            )
            self.model.save(model_path)

        return True

In [ ]:
CHECKPOINTS_DIR='./train/train_basic'
LOG_DIR='./logs/log_basic'


In [ ]:
callback=TrainAndLoggingCallback(check_freq=10000,save_path=CHECKPOINTS_DIR)


Train Model


In [ ]:
#IMPORT PPO FOR TRAININg
from stable_baselines3 import PPO

In [ ]:
#non renter environmeant
env=VizDoomGym()

In [ ]:
%pip install shimmy
%pip install tensorboard

In [ ]:
model=PPO('CnnPolicy',env,tensorboard_log=LOG_DIR,verbose=1,learning_rate=0.0001,n_steps=2048)

In [ ]:
model.learn(total_timesteps=10000,callback=callback)

In [ ]:
#activate the env in the command prompt 
#navigate to the ppo and run the command "tensorboard --logdir=." these shows the graph representation of all the selected log. 

Test The Model


In [ ]:
#import evalution policy to test the model
from stable_baselines3.common.evaluation import evaluate_policy

In [ ]:
#reload model from the disc
model=PPO.load('./train/train_basic/best_model_10000')  ## load latest model everytime

In [ ]:
#redered the environment
env=VizDoomGym(render=True)

In [ ]:
mean_reward,_=evaluate_policy(model,env,n_eval_episodes=10)

In [ ]:
mean_reward

In [ ]:
for episode in range(5):
    obs =env.reset()
    done= False
    total_reward=0
    while not done:
        action,_ =model.predict(obs)
        obs, reward, done, info =env.step(action)
        time.sleep(0.05)
        total_reward +=reward
    print('Total Reward for episode {} is {}'.format(total_reward, episode))
    time.sleep(2)